# GitHub에서 기존 YOLO + best84 파이프라인 실행

Colab **T4 GPU**에서 셀을 순서대로 실행합니다. 추가학습이나 Kaggle 자동 제출은 하지 않습니다. 원본 코드만 GitHub에서 가져오며 이미지·가중치는 기존 Drive 파일을 사용합니다.

기존 제출 기준선 CSV와 R1 비교 CSV를 함께 생성합니다. 두 파이프라인의 보고된 public LB는 각각 0.39이며, R1의 LB 향상은 확인되지 않았습니다. 재실행하면 저장된 코드 커밋과 완료 캐시를 재사용합니다. 새 커밋이나 다른 설정으로 비교하려면 새 OUTPUT을 지정하세요.

이 노트북의 구문과 구성은 검사했습니다. 새 GitHub 진입점을 통한 전체 GPU 실행은 아직 검증하지 않았습니다. 저장소가 비공개이면 Colab에서 별도 GitHub 인증이 필요합니다. 토큰을 노트북에 저장하지 마세요.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, importlib.util, json, os, signal, subprocess, sys, time

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/filament-segmentation-2026')
TEST_IMAGES = BASE/'MAGFiLO_1.0_Kaggle_2026/test/test_images'
YOLO_CHECKPOINT = Path('/content/drive/MyDrive/SolarFilament_LimbOnly_YOLO26_mixed_FullRes/best.pt')
NATIVE_CHECKPOINT = Path('/content/drive/MyDrive/SolarFilament_NativeUNet_A100/20261028T030808Z_bedf3e21/best.pt')
OUTPUT = BASE/'submissions/github_best84_t4_v1'
MAX_IMAGES = None  # 2로 빠른 점검 후 None으로 바꾸면 이어서 전체180장을 처리합니다.
REPOSITORY = 'https://github.com/ikjoo5/solar_filament.git'
REF = 'main'  # 최초 실행 때만 사용. 재개는 OUTPUT에 저장된 커밋을 고정합니다.
CODE = Path('/content/solar_filament_github')


In [ ]:
def run_git(*args):
    return subprocess.check_output(['git', *args], text=True,
        env=dict(os.environ, GIT_TERMINAL_PROMPT='0')).strip()

OUTPUT.mkdir(parents=True, exist_ok=True)
pin_path = OUTPUT/'source_commit.json'
pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
if pin and pin['repository'] != REPOSITORY:
    raise ValueError('저장된 저장소 주소가 다릅니다. 별도 OUTPUT을 사용하세요.')
if not CODE.exists():
    run_git('clone', '--no-checkout', REPOSITORY, str(CODE))
if run_git('-C', str(CODE), 'remote', 'get-url', 'origin') != REPOSITORY:
    raise ValueError('CODE 경로에 다른 저장소가 있습니다.')
# --no-checkout 직후에는 인덱스가 비어 있으므로 최초 checkout 전에 status를 확인하지 않습니다.
if (CODE/'.git/index').exists() and run_git('-C', str(CODE), 'status', '--porcelain'):
    raise ValueError('로컬 코드 변경을 보존합니다. 다른 CODE 경로를 지정하세요.')
revision = pin['commit'] if pin else REF
run_git('-C', str(CODE), 'fetch', '--depth', '1', 'origin', revision)
commit = run_git('-C', str(CODE), 'rev-parse', 'FETCH_HEAD')
if pin and commit != pin['commit']:
    raise ValueError('재개할 코드 커밋이 다릅니다.')
run_git('-C', str(CODE), 'checkout', '--detach', commit)
if pin is None:
    pending = pin_path.with_suffix('.pending')
    pending.write_text(json.dumps(dict(repository=REPOSITORY, commit=commit), indent=2))
    pending.replace(pin_path)
print('고정 실행 커밋:', commit)

spec = importlib.util.spec_from_file_location('repo_bootstrap', CODE/'tools/r1_submission_colab_bootstrap.py')
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
for protected in (TEST_IMAGES, YOLO_CHECKPOINT.parent, NATIVE_CHECKPOINT.parent):
    bootstrap.require_separate(OUTPUT, protected)
for path in (YOLO_CHECKPOINT, NATIVE_CHECKPOINT):
    if not path.is_file(): raise FileNotFoundError(str(path))
if not TEST_IMAGES.is_dir(): raise FileNotFoundError(str(TEST_IMAGES))
bootstrap.ensure_dependencies(OUTPUT)  # Colab의 torch/torchvision을 유지합니다.

names = sorted(p.name for p in TEST_IMAGES.rglob('*')
    if p.is_file() and p.suffix.lower() in ('.jpeg', '.jpg', '.png'))
expected = 'f9b2272771d3f6666f6ef812f79968a27f3d448ac2f277eda46c854ad38d598f'
encoded = json.dumps(names, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False).encode()
if len(names) != 180 or len(set(names)) != 180 or hashlib.sha256(encoded).hexdigest() != expected:
    raise ValueError('공식 test180개 파일명 목록과 다릅니다.')

def immutable_json(path, value):
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError('기존 설정을 보존합니다. 별도 OUTPUT을 사용하세요: '+str(path))
    else:
        pending = path.with_suffix('.pending')
        pending.write_text(json.dumps(value, ensure_ascii=False, indent=2))
        pending.replace(path)

names_path = OUTPUT/'official_test_names.json'
immutable_json(names_path, names)
config = dict(images=str(TEST_IMAGES), expected_names_path=str(names_path),
    yolo_checkpoint=str(YOLO_CHECKPOINT), native_checkpoint=str(NATIVE_CHECKPOINT),
    output_root=str(OUTPUT), device='cuda', native_batch=1, precision='fp16', expected_images=180)
config_path = OUTPUT/'github_config.json'
immutable_json(config_path, config)
print('설정 저장:', config_path)


In [ ]:
if MAX_IMAGES is not None and (type(MAX_IMAGES) is not int or not 1 <= MAX_IMAGES <= 180):
    raise ValueError('MAX_IMAGES는 None 또는1~180 정수입니다.')
command = [sys.executable, '-u', str(CODE/'tools/r1_test_submission.py'), '--config', str(config_path)]
if MAX_IMAGES is not None: command += ['--max-images', str(MAX_IMAGES)]
env = dict(os.environ, PYTHONUNBUFFERED='1', TORCH_CUDNN_V8_API_DISABLED='1',
           CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='1')
log_path = OUTPUT/'github_inference.log'
def tail(count=5):
    with log_path.open('rb') as stream:
        stream.seek(max(0, log_path.stat().st_size-24000))
        return '\n'.join(stream.read().decode('utf-8', 'replace').splitlines()[-count:])
print('전체 실행 로그:', log_path)
with log_path.open('ab', buffering=0) as log:
    process = subprocess.Popen(command, cwd=CODE, env=env, stdout=log,
        stderr=subprocess.STDOUT, start_new_session=True)
    try:
        while True:
            try:
                code = process.wait(timeout=30)
                break
            except subprocess.TimeoutExpired:
                print(tail(), flush=True)
        if code:
            print(tail(50))
            raise subprocess.CalledProcessError(code, command)
    finally:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
result = json.loads((OUTPUT/'latest_results.json').read_text())
print('상태:', result['status'], '완료 이미지:', result['completed_images'])
if result['status'] == 'complete':
    manifest = json.loads((OUTPUT/'run_manifest.json').read_text())
    for name, expected_sha in manifest['output_sha256'].items():
        if hashlib.sha256((OUTPUT/name).read_bytes()).hexdigest() != expected_sha:
            raise ValueError('완료 파일이 변경됐습니다: '+name)
    if hashlib.sha256(Path(result['archive']).read_bytes()).hexdigest() != result['archive_sha256']:
        raise ValueError('완료된 결과 ZIP이 변경됐습니다.')
    for key in ('baseline_csv', 'r1_csv', 'archive'):
        path = Path(result[key])
        print('결과:', path)
        try: files.download(str(path))
        except Exception: print('Drive에서 직접 받을 수 있습니다:', path)
else:
    print('부분 실행입니다. MAX_IMAGES=None으로 이 노트북을 다시 실행하세요.')
